[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-06-mt-evaluation-homework.ipynb)

# CS 479 — MT Evaluation on Your Own Data
### Lecture 6 homework, and the start of Assignment 6

The in-class notebook — *MT Evaluation (Lecture 6)* — scored data we supplied. This one
scores **yours**, and it is the genuine beginning of Assignment 6.

Do the in-class notebook first if you have not. This one assumes you know what BLEU and
chrF measure and reuses the same three functions.

**What you need:** the two cleaned, sentence-aligned files you submitted for Assignment 5.

---

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — homework for Lecture 6 (*Human and Automatic MT Evaluation*). A head start on **A6** (due at Lecture 7).

## Setup

`torchlingo` is the Matrix Lab's teaching library for neural MT. You will train models
with it later in the semester. Today we want three functions out of it — BLEU, chrF and
TER — which wrap `sacrebleu` with defaults that are sane for a classroom.

One of those defaults matters today: `compute_chrf` uses **chrF++** (`word_order=2`),
which counts word n-grams alongside character ones. Plain `sacrebleu` defaults to
`word_order=0`, and on the two sentences in Part 1 that is the difference between 78.40
and 76.97. Metric settings move numbers; this is the first example of it you will see.

In [ ]:
# Install TorchLingo when running in Colab. Nothing to uncomment.
#
# A commented-out install is the most reliable way to lose twenty minutes of a
# class: the cell "succeeds" because it does nothing, the next cell raises
# ModuleNotFoundError, and you cannot tell whether the library is broken or you
# forgot a step. So this decides for itself and says what it did.
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Colab detected. Installing torchlingo and sacrebleu ...")
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "torchlingo", "sacrebleu"],
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        print(completed.stdout[-1500:])
        print(completed.stderr[-1500:], file=sys.stderr)
        raise RuntimeError(
            "The install failed, so nothing below this cell will run. The pip output "
            "above is the reason. Most often it is a transient network error, so run "
            "this cell again before doing anything else."
        )
    print("Installed.")
else:
    print("Not in Colab, so using the torchlingo already installed here.")

In [ ]:
# Prove the setup worked. Fail here, loudly, rather than five cells from now.
from torchlingo.evaluation import compute_bleu, compute_chrf, compute_ter

import sacrebleu
import torchlingo

print(f"torchlingo  {torchlingo.__version__}")
print(f"sacrebleu   {sacrebleu.__version__}")

---
## Your own data

Upload the two cleaned, sentence-aligned files you submitted for Assignment 5 — use the
file pane on the left in Colab, or the cell below — then fill in the TODOs.

In [ ]:
from google.colab import files
uploaded = files.upload()   # pick your two files
print(list(uploaded))


In [ ]:
# TODO: point these at your two uploaded filenames.
SRC_FILE = ''   # your source-language file (English, if you translate out of English)
TGT_FILE = ''   # your target-language file, line-for-line aligned with SRC_FILE

with open(SRC_FILE, encoding='utf-8') as f:
    src_lines = [l.rstrip('\n') for l in f]
with open(TGT_FILE, encoding='utf-8') as f:
    tgt_lines = [l.rstrip('\n') for l in f]

assert len(src_lines) == len(tgt_lines), (len(src_lines), len(tgt_lines))
print(f'{len(src_lines):,} aligned pairs')


In [ ]:
# Take a random sample. Ten is enough for class; the assignment wants at least 500.
import random
random.seed(479)

N = 10
idx = random.sample(range(len(src_lines)), N)
sample_src = [src_lines[i] for i in idx]
sample_ref = [tgt_lines[i] for i in idx]

for s in sample_src:
    print(s)


### Translate them

Copy the ten source sentences printed above into an MT system in another tab
(translate.google.com, bing.com/translator, deepl.com, translate.yandex.com), then
paste the ten translations back below, **in the same order**.

For the assignment you will automate this for 500 or 1000 sentences. For the next ten
minutes, copy and paste is faster than an API key.


In [ ]:
# TODO: paste the machine translations here, one per line, same order as above.
hypotheses = """

""".strip().split('\n')

assert len(hypotheses) == N, f'got {len(hypotheses)}, expected {N}'


In [ ]:
# TODO: score your sample against your own target sentences as references.
b = compute_bleu(hypotheses, sample_ref)
c = compute_chrf(hypotheses, sample_ref)
t = compute_ter(hypotheses, sample_ref)

print('BLEU', round(b.score, 2))
print('chrF', round(c.score, 2))
print('TER ', round(t.score, 2), '(lower is better)')


In [ ]:
# Now rank them yourself, before you look at the numbers again.
# Print each source, your reference, and the MT output side by side.
for s, r, h in zip(sample_src, sample_ref, hypotheses):
    print('SRC:', s)
    print('REF:', r)
    print('MT :', h)
    print('-' * 70)


### If your language is not written with spaces

Chinese, Japanese, Thai, Khmer, Lao, Burmese: word-level BLEU is close to meaningless
for you, because the tokenizer has to guess where the words are. `compute_bleu` detects
this and switches to character tokenization, but chrF is the metric to trust. Say so in
your Assignment 6 analysis; it is exactly the kind of observation the analysis is for.


---
## What to record

Write these down; they are the opening of your Assignment 6 analysis:

1. **Two numbers**: the BLEU and the chrF you got on your sample.
2. **One sentence**: whether you agree with the ranking those numbers imply, having read
   the translations yourself.
3. **One count**: if your BLEU came out at or near zero, how many sentences did it take
   before the number started to move? Rerun with a larger `N`.

A BLEU of zero on ten short sentences is the expected answer, not a bug.

---

### Where this goes next

Assignment 6 is this at scale, plus the piece no library can do for you: a human ranking
of 50 sentences in **MTEval**, done *before* you see any score, and a written analysis of
where the metrics and your own judgment parted company.